# nanoGenRec Public MovieLens GPU Benchmark

This notebook runs the redistributable nanoGenRec public benchmark on Google Colab GPU. It uses public MovieLens data only: ratings and metadata -> semantic IDs -> tiny NTP training -> SID-constrained full-recall evaluation.

Before running: choose `Runtime` -> `Change runtime type` -> `T4 GPU`.

In [ ]:
!nvidia-smi

## Clone nanoGenRec

Re-running this cell updates an existing checkout.

In [ ]:
import os
import subprocess
from pathlib import Path

repo = Path("/content/nanoGenRec")
if repo.exists():
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "https://github.com/yzq986/nanoGenRec.git", str(repo)], check=True)
os.chdir(repo)
print(f"Working directory: {Path.cwd()}")


## Recommended T4 run

This is the recommended first public GPU run. It is intentionally stronger than the CPU smoke test while staying small enough for a free T4 session.

In [ ]:
import json
import subprocess
import time
from pathlib import Path

output_dir = Path("public_benchmarks/runs/ml-latest-small-colab-t4")
cmd = [
    "python", "run.py", "public-movielens",
    "--dataset", "ml-latest-small",
    "--output_dir", str(output_dir),
    "--min_rating", "5.0",
    "--min_user_items", "5",
    "--max_users", "0",
    "--max_items_per_user", "100",
    "--feature_source", "hybrid",
    "--collab_window", "5",
    "--clusters", "16,16,16",
    "--feature_dim", "96",
    "--kmeans_iters", "5",
    "--kmeans_sample_size", "2048",
    "--train_mode", "sliding",
    "--min_context_items", "2",
    "--embed_dim", "96",
    "--n_heads", "4",
    "--layers", "2",
    "--batch_size", "128",
    "--epochs", "8",
    "--eval_samples", "1000",
    "--beam_size", "1000",
    "--max_seq_len", "128",
    "--device", "cuda",
]
start = time.time()
subprocess.run(cmd, check=True)
elapsed = time.time() - start
gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,memory.used", "--format=csv,noheader"],
    text=True,
    capture_output=True,
    check=False,
)
runtime = {"elapsed_seconds": elapsed, "gpu": gpu.stdout.strip(), "command": cmd}
(output_dir / "runtime.json").write_text(json.dumps(runtime, indent=2))
print(json.dumps(runtime, indent=2))


## Inspect metrics

In [ ]:
import json
from pathlib import Path

run_dir = Path("public_benchmarks/runs/ml-latest-small-colab-t4")
metrics = json.loads((run_dir / "metrics.json").read_text())
print(json.dumps(metrics, indent=2))
runtime_path = run_dir / "runtime.json"
if runtime_path.exists():
    print(json.dumps(json.loads(runtime_path.read_text()), indent=2))

## Optional: larger run

If the T4 session is stable, try MovieLens 1M. This is the next useful public scale. If Colab disconnects, reduce `--epochs`, `--beam_size`, or `--eval_samples` first.

In [ ]:
# !python run.py public-movielens \
#     --dataset ml-1m \
#     --output_dir public_benchmarks/runs/ml-1m-colab-t4 \
#     --min_rating 4.0 \
#     --min_user_items 10 \
#     --max_users 0 \
#     --max_items_per_user 100 \
#     --feature_source hybrid \
#     --collab_window 5 \
#     --clusters 64,64,64 \
#     --feature_dim 128 \
#     --kmeans_iters 5 \
#     --kmeans_sample_size 8192 \
#     --train_mode sliding \
#     --min_context_items 2 \
#     --embed_dim 128 \
#     --n_heads 4 \
#     --layers 3 \
#     --batch_size 256 \
#     --epochs 5 \
#     --eval_samples 1000 \
#     --beam_size 1000 \
#     --max_seq_len 128 \
#     --device cuda


## Optional: save outputs to Google Drive

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/nanoGenRec-public
# !cp -r public_benchmarks/runs/ml-latest-small-colab-t4 /content/drive/MyDrive/nanoGenRec-public/